# overlays

> Popovers, menus, dialogs, drawers, and tooltips

In [ ]:
#| default_exp overlays

An overlay sits above the page without changing its layout. Popovers provide light dismiss; dialogs make the page behind them inert. We keep those native behaviours, add a shared floating surface, and position popovers beside the control that opened them. Menus and tooltips build on that panel.

In [ ]:
from fasthtml.common import Div, Button, A, P, Small, H2, Hr, FT, unqid
from answerui.registry import reg, hypelt
from answerui.layout import Stack, Cluster
from answerui.demo import example


In [ ]:
from fasthtml.common import *
from answerui.demo import demo


## Popover

`Popover` is a floating neutral surface. A button opens it with `popovertarget`, and the browser supplies light dismiss and Escape. While the panel is open, the button that opened it carries `aria-expanded`, so it shows as pressed. The `anchored` marker keeps its styling separate from other uses of the native `popover` attribute.

One positioning rule serves every browser: place the panel below its actual invoker, use the space above when necessary, and keep it inside the viewport. An ancestor marked `anchor`, such as a navbar, stands in for the button in the vertical placement, so a menu opens below the bar rather than across its padding. While open, it follows scrolling, resizing, and changes in content size. Closing the panel removes those listeners. The placement is scripted, as the popover's `placement` rule, until CSS anchor positioning is available in every engine.

In [ ]:
_popover_on = {
    ('button[popovertarget], button[commandfor]', 'click'): r"""
const id = el.getAttribute('popovertarget') || el.getAttribute('commandfor');
const panel = document.getElementById(id);
if (panel?.matches('[popover][anchored]')) panel._anchor = el;""",
    ('&', 'beforetoggle', True): r"""
if (e.target !== el) return;
if (e.newState === 'open') {
    if (e.source) el._anchor = e.source;
    el.style.visibility = 'hidden';
} else el._unwatch?.();""",
    ('&', 'toggle', True): r"""
if (e.target !== el) return;
const opener = el._anchor, invoked = opener && [opener.getAttribute('popovertarget'), opener.getAttribute('commandfor')].includes(el.id);
if (invoked) opener.setAttribute('aria-expanded', String(el.matches(':popover-open')));
el._unwatch?.();
if (!el.matches(':popover-open')) return;
const anchor = el._anchor;
const place = () => {
    if (!anchor?.isConnected) { el.hidePopover(); return; }
    const r = anchor.getBoundingClientRect(), b = (anchor.closest('[anchor]') ?? anchor).getBoundingClientRect(), gap = 4, edge = 8;
    const vw = document.documentElement.getBoundingClientRect().width, vh = document.documentElement.clientHeight;
    const below = vh - b.bottom - gap - edge, above = b.top - gap - edge;
    const height = el.scrollHeight + el.offsetHeight - el.clientHeight;
    const up = height > below && above > below;
    el.style.setProperty('--_available-height', Math.max(0, up ? above : below) + 'px');
    const x = getComputedStyle(anchor).direction === 'rtl' ? r.right - el.offsetWidth : r.left;
    const y = up ? b.top - gap - el.offsetHeight : b.bottom + gap;
    el.style.left = Math.max(edge, Math.min(x, vw - el.offsetWidth - edge)) + 'px';
    el.style.top = Math.max(edge, Math.min(y, vh - el.offsetHeight - edge)) + 'px';
};
if (anchor) {
    place();
    const watch = new AbortController(), size = new ResizeObserver(place);
    addEventListener('resize', place, {signal: watch.signal});
    document.addEventListener('scroll', place, {capture: true, signal: watch.signal});
    size.observe(anchor); size.observe(el);
    el._unwatch = () => { watch.abort(); size.disconnect(); };
}
el.style.visibility = '';""",
    ('&', ':before-cleanup-element'): "if (e.target === el) el._unwatch?.()"}

_popover_css = r"""
[popover][anchored] {
  --face: var(--n-2); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space); box-shadow: var(--floating);
  position: fixed; inset: auto; margin: 0; max-inline-size: calc(100vw - 1rem); overflow: auto;
  max-block-size: var(--_available-height, calc(100dvh - 1rem));
  opacity: 0;
  transition: opacity var(--motion) var(--ease), display var(--motion) allow-discrete, overlay var(--motion) allow-discrete;
}
[popover][anchored]:not(:popover-open) { display: none; }
[popover][anchored]:popover-open { opacity: 1; @starting-style { opacity: 0; } }
"""

@hypelt(sel='[popover][anchored]', layer='components', css=_popover_css)
def Popover(*c, popover='auto', **attrs):
    "A floating panel positioned beside the button that opens it"
    return Div(*c, popover=popover, anchored=True, **attrs)

Popover.rule('placement', on=_popover_on, layer='components', doc='Remembers the button that opened the panel, places the panel beside it within the viewport less any scrollbar or its gutter, and follows scrolling and resizing until it closes.')

Popover.placement [components] [popover][anchored]: Remembers the button that opened the panel, places the panel beside it, and follows scrolling and resizing until it closes.

Any button can open a popover by naming its id. The panel holds ordinary content:

In [ ]:
@example(Popover)
def _popover_example():
    id = unqid()
    return Cluster(Button('What is a workspace?', type='button', secondary=True, popovertarget=id),
        Popover(P("A workspace holds a team's projects, members, and billing."), id=id, style='max-inline-size: 18rem'))

In [ ]:
Popover.demo(height='9rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Dropdown

`Dropdown` returns a trigger and its popover of links, buttons, and separators. Other content, such as a heading naming the signed-in user, sits in line with the items. Give it text for a secondary trigger, or a configured `Button` to choose the appearance, accessible name, and other attributes. A trigger that shows only an avatar or an icon needs a name of its own, so pass a `Button` with an `aria-label`. The trigger is always `type='button'`, so opening it does not submit a surrounding form.

These are ordinary HTML controls, not an ARIA application menu: Tab still visits the items normally. Arrow keys, Home, and End also move between enabled, visible items. Opening focuses the first item; choosing an action closes the panel, including actions handled by htmx. Escape and light dismiss remain native. The keyboard handling and the closing are the dropdown's `keys` and `choose` rules.

In [ ]:
_dropdown_keys = {
    ('&', 'toggle', True): r"""
if (e.target !== el || !el.matches(':popover-open')) return;
[...el.querySelectorAll('a[href], button:not(:disabled)')]
    .find(x => x.getAttribute('aria-disabled') !== 'true' && x.getClientRects().length && getComputedStyle(x).visibility === 'visible')?.focus();""",
    ('&', 'keydown'): r"""
if (!['ArrowDown', 'ArrowUp', 'Home', 'End'].includes(e.key)) return;
const items = [...el.querySelectorAll('a[href], button:not(:disabled)')]
    .filter(x => x.getAttribute('aria-disabled') !== 'true' && x.getClientRects().length && getComputedStyle(x).visibility === 'visible');
if (!items.length) return;
e.preventDefault();
const i = items.indexOf(document.activeElement);
const next = e.key === 'Home' ? 0 : e.key === 'End' ? items.length - 1
    : (i + (e.key === 'ArrowDown' ? 1 : -1) + items.length) % items.length;
items[next].focus();"""}

_dropdown_choose = {
    ('& > [aria-disabled=true]', 'click', True): "e.preventDefault(); e.stopImmediatePropagation()",
    ('&', 'click'): r"""
const item = e.target.closest('a[href], button');
if (!item || !el.contains(item)) return;
if (item.matches(':disabled, [aria-disabled=true]')) { e.preventDefault(); return; }
el.hidePopover();"""}

_dropdown_css = r"""
[popover][dropdown] { min-inline-size: min(12rem, calc(100vw - 1rem)); padding: var(--space-3); }
[popover][dropdown]:popover-open { display: grid; align-content: start; block-size: max-content; }
[dropdown] > :is(a, button) {
  display: block; inline-size: 100%; text-align: start; padding: var(--space-2) var(--space-1);
  font: inherit; font-weight: var(--weight); color: inherit; text-decoration: none;
  background: none; border: 0; border-radius: var(--radius-1); box-shadow: none;
}
[dropdown] > :is(a, button):not(:disabled, [aria-disabled=true]):is(:hover, :focus-visible) {
  background: var(--v-3); color: var(--v2);
}
[dropdown] > :is(a, button)[danger]:not(:disabled, [aria-disabled=true]) { color: var(--v); }
[dropdown] > :is(a, button):is(:disabled, [aria-disabled=true]) { color: var(--n1); cursor: default; }
[dropdown] > [hidden] { display: none; }
[dropdown] > hr { margin-block: var(--space-3); }
[dropdown] > :not(a, button, hr) { padding-inline: var(--space-1); }
"""

@hypelt(sel='[popover][dropdown]', layer='components', css=_dropdown_css)
def Dropdown(
    trigger,  # Label text or a configured Button
    *items,   # Buttons, links, Hr separators, and other content such as a heading, which aligns with the items
    id=None,  # Panel id; generated when omitted
    **attrs,  # Attributes for the panel
):
    "A button and its popover of ordinary links and actions"
    id = id or unqid()
    if not isinstance(trigger, FT) or trigger.tag != 'button': trigger = Button(trigger, secondary=True)
    return trigger(type='button', popovertarget=id), Popover(*items, dropdown=True, id=id, **attrs)

Dropdown.rule('keys', on=_dropdown_keys, layer='components', doc='Opening focuses the first item; the arrow keys, Home, and End move between enabled, visible items.')
Dropdown.rule('choose', on=_dropdown_choose, layer='components', doc='Choosing an item closes the dropdown; an item marked aria-disabled does nothing.')

Dropdown.choose [components] [popover][dropdown]: Choosing an item closes the dropdown; an item marked aria-disabled does nothing.

The account menu opens with the user's name and email, in line with its items, and uses the default trigger; Options supplies its own button. Try the arrow keys: the heading is skipped, and so is the disabled Duplicate action.

In [ ]:
@example(Dropdown)
def _dropdown_example():
    trigger, panel = Dropdown(Button('Options', aria_label='Project actions'),
        Button('Rename', type='button'), Button('Duplicate', type='button', disabled=True),
        Hr(), Button('Delete', type='button', danger=True))
    assert trigger.get('aria-label') == 'Project actions'
    return Cluster(Dropdown('Account', P('Ada Lovelace'), Small('ada@example.com'), Hr(), A('Profile', href='#'), A('Billing', href='#'),
        Hr(), Button('Sign out', type='button')), trigger, panel)

In [ ]:
Dropdown.demo(height='14rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Invoker commands

`commandfor` names a target and `command` says what to do: `show-modal`, `close`, or `toggle-popover`. Where the browser lacks invoker commands, one delegated rule calls the equivalent method. There is no constructor to learn.

In [ ]:
reg.rule('commands', sel='button[commandfor]', on={('&', 'click'): r"""
if ('commandForElement' in el) return;
const t = document.getElementById(el.getAttribute('commandfor')), c = el.getAttribute('command');
if (!t) return;
if (!['show-modal', 'close', 'toggle-popover'].includes(c)) return;
e.preventDefault();
if (c === 'show-modal') t.showModal();
else if (c === 'close') t.close();
else { t._anchor = el; t.togglePopover(); }"""},
    layer='components', doc='`command` and `commandfor` on engines without invoker commands.')

commands [components] button[commandfor]: `command` and `commandfor` on engines without invoker commands.

## Dialog

A modal is the platform's `dialog`, opened with `show-modal`. Focus moves into it, Escape closes it, and the page behind is inert, with nothing for us to script. The panel is the floating surface at the larger radius, and the backdrop is the ink at low alpha. Both fade over `--motion`. A dialog inherits the page's font and colours through its DOM parent, so it restates nothing. Closing is a button with `command="close"`, or a form with `method=\"dialog\"`. A click on the backdrop closes nothing unless the dialog carries `closedby=\"any\"`, which suits a drawer more than a confirmation:

In [ ]:
Dialog = hypelt('Dialog', layer='components', doc='A modal panel over an inert page; `drawer` slides it in from an edge.', css=r"""
dialog {
  --face: var(--n-2); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius1); padding: var(--space1); box-shadow: var(--floating);
  inline-size: min(32rem, calc(100% - 2 * var(--space)));
  opacity: 0; translate: 0 var(--space-2);
  transition: opacity var(--motion) var(--ease), translate var(--motion) var(--ease), display var(--motion) allow-discrete, overlay var(--motion) allow-discrete;
}
dialog[open] { opacity: 1; translate: none; @starting-style { opacity: 0; translate: 0 var(--space-2); } }
dialog::backdrop { background: oklch(from var(--n2) l c h / .4); opacity: 0;
  transition: opacity var(--motion) var(--ease), display var(--motion) allow-discrete, overlay var(--motion) allow-discrete; }
dialog[open]::backdrop { opacity: 1; @starting-style { opacity: 0; } }
dialog:not([open]) { display: none; }
""")
Dialog.variant('drawer', r"""
--_slide: 100%; --drawer-size: 24rem; inset: 0; inset-inline-start: auto;
block-size: 100dvh; max-block-size: none; inline-size: var(--drawer-size); max-inline-size: none; margin: 0;
border-radius: 0; border-block: 0; border-inline-end: 0; translate: var(--_slide) 0;
&:dir(rtl) { --_slide: -100%; }
&[drawer=start] { --_slide: -100%; inset-inline: 0 auto; border-inline: 0; border-inline-end: 1px solid var(--n-1); }
&[drawer=start]:dir(rtl) { --_slide: 100%; }
&[open] { translate: none; @starting-style { translate: var(--_slide) 0; } }
@media (width < 40rem) { &, &[drawer=start] { inline-size: 100%; border: 0; } }
""", layer='components', doc='Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem; `start` slides in from the other edge.')

Dialog.drawer [components] :is(dialog)[drawer]: Slides in from the end edge and runs the full height, `--drawer-size` wide, or the whole screen below 40rem; `start` slides in from the other edge.

A dialog that htmx fills opens itself. A button that targets the dialog's body with `hx-get` needs no `command`, and the dialog shows nothing stale: it opens once the new content has arrived. A swap of the dialog itself, such as a closed one sent out of band to close it, leaves it closed.

In [ ]:
Dialog.rule('fill', on={('&', ':after-swap'): "if (e.target !== el && !el.open) el.showModal()"}, layer='components',
    doc='A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.')

Dialog.fill [components] dialog: A closed dialog opens when htmx swaps content into it; a swap of the dialog itself leaves it as it is.

A confirmation with its trigger. `aria-labelledby` connects the dialog to its heading; simply putting a heading inside a dialog does not name it. The content is an ordinary stack with a cluster of buttons.

In [ ]:
@example(Dialog)
def _dialog_example():
    id, title = unqid(), unqid()
    return Cluster(Button('Delete project', type='button', danger=True, command='show-modal', commandfor=id),
        Dialog(Stack(H2('Delete this project?', id=title), P('Everything in it is removed. This cannot be undone.'),
            Cluster(Button('Cancel', type='button', secondary=True, command='close', commandfor=id, autofocus=True),
                Button('Delete', type='button', danger=True, command='close', commandfor=id), gap='var(--space-2)')),
            id=id, aria_labelledby=title))

In [ ]:
Dialog.demo(height='16rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Drawer

A drawer is a dialog that slides in from an edge and runs the full height: `drawer=True` for the right edge, `drawer='start'` for the left. It is `--drawer-size` wide, 24rem unless the app sets the token, and below 40rem it fills the screen, since a sliver of page beside it is no use on a phone. Everything else, opening, closing, the backdrop, and the inert page, is the dialog's. A drawer also closes on a click outside it, with `closedby=\"any\"`, in browsers that support that attribute; elsewhere its button still closes it:

In [ ]:
demo(Cluster(Button('Filters', type='button', secondary=True, command='show-modal', commandfor='filters'),
    Button('Navigation', type='button', secondary=True, command='show-modal', commandfor='nav-drawer'),
    Dialog(Stack(H2('Filters', id='filters-title'), P('Narrow the list.'),
        Button('Done', type='button', command='close', commandfor='filters')),
        id='filters', aria_labelledby='filters-title', drawer=True, closedby='any'),
    Dialog(Stack(H2('Navigation', id='nav-title'), P('Sections of the app.'),
        Button('Close', type='button', secondary=True, command='close', commandfor='nav-drawer')),
        id='nav-drawer', aria_labelledby='nav-title', drawer='start', closedby='any')), height='12rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Tooltip

`Tooltip` pairs a control with a short, supplementary description. It adds `aria-describedby` without replacing an existing description, and returns both elements for normal FastHTML composition. The control still needs its own name; a tooltip is not a substitute for a label.

The description is real markup in a manual popover, not a pseudo-element borrowed from the control. It wraps, stays inside the viewport, and can be hovered. Focus or hover keeps it open; Escape dismisses it. A short delay bridges the gap between the control and the description. Put interactive content in a `Popover`, not a tooltip. The showing and hiding is the tooltip's `show` rule.

Moving from one control to the next swaps their tooltips at once, with no fade between them, as the platform's own tooltips do.

In [ ]:
_tooltip_on = {
    ('&', ':load'): r"""
el._tipHide = () => {
    clearTimeout(el._tipTimer);
    el._tipTimer = setTimeout(() => {
        if (!el.matches(':popover-open') || el.matches(':hover')) return;
        if (el._anchor?.matches(':hover, :focus-within')) return;
        el.hidePopover();
    }, 150);
};
el._tipShow = anchor => {
    clearTimeout(el._tipTimer); el._anchor = anchor;
    if (el.matches(':popover-open')) return;
    const others = [...document.querySelectorAll('[role=tooltip]:popover-open')];
    for (const t of others) { clearTimeout(t._tipTimer); t.style.transition = 'none'; t.hidePopover(); }
    if (others.length) el.style.transition = 'none';
    el.showPopover();
    requestAnimationFrame(() => { for (const t of [...others, el]) t.style.transition = ''; });
};""",
    ('[tooltip]', 'pointerover'): "if (!el.contains(e.relatedTarget)) document.getElementById(el.getAttribute('tooltip'))._tipShow?.(el)",
    ('[tooltip]', 'focusin'): "document.getElementById(el.getAttribute('tooltip'))._tipShow?.(el)",
    ('[tooltip]', 'pointerout'): "if (!el.contains(e.relatedTarget)) document.getElementById(el.getAttribute('tooltip'))._tipHide?.()",
    ('[tooltip]', 'focusout'): "document.getElementById(el.getAttribute('tooltip'))._tipHide?.()",
    ('&', 'pointerover'): "clearTimeout(el._tipTimer)",
    ('&', 'pointerout'): "if (!el.contains(e.relatedTarget)) el._tipHide?.()",
    ('body', 'keydown'): r"""
if (e.key !== 'Escape') return;
const tips = document.querySelectorAll('[role=tooltip]:popover-open');
if (tips.length) e.preventDefault();
for (const tip of tips) { clearTimeout(tip._tipTimer); tip.hidePopover(); }""",
    ('&', ':before-cleanup-element'): "if (e.target === el) clearTimeout(el._tipTimer)"}

_tooltip_css = r"""
[popover][role=tooltip] {
  --face: var(--n2); background: var(--face); color: var(--n-3); border: 0;
  padding: var(--space-3) var(--space-2); border-radius: var(--radius-1);
  inline-size: max-content; max-inline-size: min(20rem, calc(100vw - 1rem)); overflow-wrap: anywhere;
  font-size: var(--text-1); font-weight: var(--weight); line-height: var(--leading);
}
"""

@hypelt(sel='[role=tooltip]', layer='components', css=_tooltip_css)
def Tooltip(
    trigger, # A control with its own accessible name
    text,    # Supplementary, non-interactive content
    id=None, # Description id; generated when omitted
    **attrs, # Attributes for the description
):
    "A control and its supplementary description, shown on hover or focus"
    id = id or unqid()
    describedby = ' '.join(filter(None, [trigger.get('aria-describedby'), id]))
    return trigger(tooltip=id, aria_describedby=describedby), Popover(text, popover='manual', role='tooltip', id=id, **attrs)

Tooltip.rule('show', on=_tooltip_on, layer='components', doc='Shows on hover or focus of the control, hides after a short delay once both are gone, swaps at once between controls, and closes on Escape.')


Tooltip.show [components] [role=tooltip]: Shows on hover or focus of the control, hides after a short delay once both are gone, swaps at once between controls, and closes on Escape.

In [ ]:
@example(Tooltip)
def _tooltip_example():
    return Cluster(Tooltip(Button('Archive', type='button', secondary=True), 'Move to the archive'),
        Tooltip(Button('Share', type='button', secondary=True), 'Copy a link'),
        Tooltip(Button('Delete', type='button', danger=True), 'This cannot be undone'))

In [ ]:
Tooltip.demo(height='7rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()